# Tensor-native N4 bias field correction

Correct a smooth multiplicative intensity bias in a uniform synthetic image.
This offline example is adapted from
`tests/bspline_flows/test_n4_bias_field_correction.py`.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
from antstorch.bspline_flows import ImageDomain, n4_bias_field_correction_tensor

domain = ImageDomain((24, 20))  # Physical axis order: x, y.
y = torch.linspace(-1, 1, 20, device=device)[:, None]
x = torch.linspace(-1, 1, 24, device=device)[None, :]
true_bias = torch.exp(0.4 * x + 0.2 * y)
image = true_bias[None, None]  # Tensor axis order: batch, channel, y, x.
corrected = n4_bias_field_correction_tensor(
    image, domain, shrink_factor=1,
    convergence={"iters": [10], "tol": 0.0},
    spline_param=(2, 2), number_of_histogram_bins=32,
)

## Measure the remaining nonuniformity
For this uniform phantom, the coefficient of variation should decrease. This
check does not establish accuracy on anatomical images with multiple tissues.

In [ ]:
before_cv = (image.std() / image.mean()).item()
after_cv = (corrected.std() / corrected.mean()).item()
assert corrected.shape == image.shape
assert torch.isfinite(corrected).all()
assert after_cv < 0.6 * before_cv
print(f"Coefficient of variation: {before_cv:.4f} -> {after_cv:.4f}")
estimated_bias = image / corrected
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, values, title in zip(axes, [image, corrected, estimated_bias],
                             ["Biased image", "Corrected image", "Estimated bias"]):
    view = ax.imshow(values[0, 0].detach().cpu(), origin="lower", cmap="gray")
    ax.set_title(title)
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

For real MRI, choose a suitable mask, resolution, spline spacing and convergence schedule.